In [1]:
!pip install pandas

  Using cached pandas-3.0.5-cp311-cp311-win_amd64.whl.metadata (19 kB)
  Using cached numpy-2.4.6-cp311-cp311-win_amd64.whl.metadata (6.6 kB)
  Using cached tzdata-2026.3-py2.py3-none-any.whl.metadata (1.4 kB)
Using cached pandas-3.0.5-cp311-cp311-win_amd64.whl (10.0 MB)
Using cached numpy-2.4.6-cp311-cp311-win_amd64.whl (12.6 MB)
Using cached tzdata-2026.3-py2.py3-none-any.whl (348 kB)



[notice] A new release of pip is available: 24.0 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
import pandas
import re

year = 2020

In [11]:
data = pandas.read_csv(f"data/{year}/{year}.csv")
data.info()

<class 'pandas.DataFrame'>
RangeIndex: 34344 entries, 0 to 34343
Data columns (total 25 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   _id                     34344 non-null  int64  
 1   DATA                    34344 non-null  str    
 2   HORA                    34344 non-null  str    
 3   CONCESSIONARIA          34344 non-null  str    
 4   RODOVIA                 34344 non-null  str    
 5   KM                      34344 non-null  float64
 6   SENTIDO                 34344 non-null  str    
 7   LATITUDE                34344 non-null  float64
 8   LONGITUDE               34344 non-null  float64
 9   CLASSE                  34344 non-null  str    
 10  SUBCLASSE               34344 non-null  str    
 11  CAUSA_PROVAVEL          34344 non-null  str    
 12  VITIMA_ILESA            34344 non-null  int64  
 13  VITIMA_LEVE             34344 non-null  int64  
 14  VITIMA_MODERADA         34344 non-null  int64  
 

In [12]:
data = data.sort_values(by=[data.DATA.name, data.HORA.name], ascending=True)

data.loc[(data.RODOVIA.str.contains("330")), :].drop(columns=["_id","VITIMA_ILESA","VITIMA_LEVE","VITIMA_MODERADA","VITIMA_GRAVE","VITIMA_FATAL","VITIMAS_SEM_INFO","VISIBILIDADE","CONDICAO_METERIOLOGICA"]).to_csv(f"data/{year}/p{year}.csv", index=False)

In [13]:
print(data["CLASSE"].unique())

<StringArray>
[                         'CHOQUE',                     'CAPOTAMENTO',
         'ATROPELAMENTO DE ANIMAL',                         'COLISÃO',
                      'TOMBAMENTO',                   'NÃO INFORMADO',
                           'QUEDA',       'ATROPELAMENTO DE PEDESTRE',
                   'ENGAVETAMENTO', 'OBJETO LANÇADO CONTRA O VEÍCULO']
Length: 10, dtype: str


In [14]:
print(list(data["SUBCLASSE"].unique()))

['CHOQUE-TALUDE OU BARRANCO', 'CAPOTAMENTO', 'CHOQUE-DEFENSA METÁLICA', 'ATROP. ANIMAL', 'COLISÃO-LATERAL', 'CHOQUE-SINALIZAÇÃO', 'TOMBAMENTO', 'CHOQUE', 'CHOQUE-ELEMENTO DE DRENAGEM', 'COLISÃO-TRANSVERSAL', 'COLISÃO-TRASEIRA', 'CHOQUE-DISPOSITIVO DE CONTENÇÃO', 'CHOQUE-POSTE', 'COM DEFENSA, BARREIRA OU SUBMARINO', 'CHOQUE-OUTROS', 'CHOQUE-BARREIRA DE CONCRETO', 'LATERAL', 'CHOQUE-ÁRVORE', '(OUTROS)', 'CHOQUE -  DEFENSA, BARREIRA OU "SUBMARINO"', 'QUEDA-VEÍCULO EM RIBANCEIRA,PONTE OU VIADUTO', 'CHOQUE - DEFENSA, BARREIRA', 'TOMBAMENTO-MOTO', 'COLISÃO-FRONTAL', 'DE PEDESTRE (OUTROS)', 'CHOQUE-VEÍCULO PARADO NA PISTA', 'ATROP. PEDESTRE', 'ATROP. PEDESTRE-ANDARILHO', 'DE ANIMAL', 'CHOQUE - DEFENSA, BARREIRA OU "SUBMARINO"', 'QUEDA-MOTO', 'CHOQUE-VEÍCULO PARADO NO ACOSTAMENTO', 'CHOQUE-MEIO FIO/CALÇAMENTO', 'CHOQUE-CERCAS,ALAMBRADOS,MOURÃO', 'CHOQUE-DEFENSA,BARREIRA OU SUBMARINO', 'ENGAVETAMENTO', 'TRASEIRA', 'OBJETO LANÇADO CONTRA VEÍCULO', 'TOMBAMENTO-BICICLETA', 'COM ELEMENTO DE DRENAGE

In [17]:
def normalizar_subclasse(s):
    if pandas.isna(s):
        return "NÃO INFORMADO"
    s = s.upper().strip()
    
    regras = [
        (r"DEFENSA|BARREIRA|SUBMARINO", "CHOQUE-DEFENSA/BARREIRA"),
        (r"DRENAGEM", "CHOQUE-ELEMENTO DE DRENAGEM"),
        (r"TALUDE|BARRANCO|CORTE", "CHOQUE-TALUDE/BARRANCO"),
        (r"MEIO.?FIO|CALÇAMENTO", "CHOQUE-MEIO FIO"),
        (r"ÁRVORE|ARVORE", "CHOQUE-ÁRVORE"),
        (r"POSTE", "CHOQUE-POSTE"),
        (r"BURACO", "CHOQUE-BURACO"),
        (r"OBJETO.*PISTA|OBJETO SOBRE A VIA|VEÍCULO PARADO NA PISTA", "CHOQUE-OBJETO NA PISTA"),
        (r"VEÍCULO PARADO NO ACOSTAMENTO", "CHOQUE-VEÍCULO PARADO NO ACOSTAMENTO"),
        (r"OAE|PILAR|VIADUTO|PONTE", "CHOQUE-OAE (PONTE/VIADUTO)"),
        (r"PRAÇA|CABINE|CANCELA|PEDÁGIO", "CHOQUE-PRAÇA DE PEDÁGIO"),
        (r"SINALIZAÇÃO|EQUIPAMENTO|PAINEL", "CHOQUE-SINALIZAÇÃO/EQUIPAMENTO"),
        (r"CERCA|ALAMBRADO|MOURÃO", "CHOQUE-CERCAS/ALAMBRADOS"),
        (r"EDIFICAÇÃO|ILHA|MATACÃO|OUTROS|NÃO IDENTIF", "CHOQUE-OUTROS"),
        
        (r"^FRONTAL$|COLIS.O-FRONTAL", "COLISÃO-FRONTAL"),
        (r"^TRASEIRA$|COLIS.O-TRASEIRA", "COLISÃO-TRASEIRA"),
        (r"^LATERAL$|COLIS.O-LATERAL", "COLISÃO-LATERAL"),
        (r"^TRANSVERSAL$|COLIS.O-TRANSVERSAL", "COLISÃO-TRANSVERSAL"),
        
        (r"^TOMBAMENTO$", "TOMBAMENTO"),
        (r"TOMBAMENTO-MOTO", "TOMBAMENTO-MOTO"),
        (r"TOMBAMENTO.*PESAD", "TOMBAMENTO-VEÍCULO PESADO"),
        (r"TOMBAMENTO-BICICLETA", "TOMBAMENTO-BICICLETA"),
        
        (r"^CAPOTAMENTO$", "CAPOTAMENTO"),
        (r"^ENGAVETAMENTO$", "ENGAVETAMENTO"),
        
        (r"SUICID|SUICÍD", "ATROP. PEDESTRE-SUICÍDIO"),
        (r"CICLISTA", "ATROP. PEDESTRE-CICLISTA"),
        (r"ATROP\.? PEDESTRE|DE PEDESTRE|PEDESTRE USUÁRIO", "ATROP. PEDESTRE-OUTROS"),
        
        (r"ANIMAL.*SILVESTRE.*GRANDE", "ATROP. ANIMAL-SILVESTRE GRANDE"),
        (r"ANIMAL.*SILVESTRE.*M.DIO", "ATROP. ANIMAL-SILVESTRE MÉDIO"),
        (r"ANIMAL.*SILVESTRE.*PEQUENO", "ATROP. ANIMAL-SILVESTRE PEQUENO"),
        (r"ANIMAL.*DOM.STICO.*GRANDE", "ATROP. ANIMAL-DOMÉSTICO GRANDE"),
        (r"ANIMAL.*DOM.STICO.*M.DIO", "ATROP. ANIMAL-DOMÉSTICO MÉDIO"),
        (r"ANIMAL.*DOM.STICO.*PEQUENO", "ATROP. ANIMAL-DOMÉSTICO PEQUENO"),
        (r"ANIMAL", "ATROP. ANIMAL-OUTROS"),
        
        (r"^QUEDA-MOTO$", "QUEDA-MOTO"),
        (r"^QUEDA-CICLISTA$", "QUEDA-CICLISTA"),
        (r"RIBANCEIRA|EM RIBANCEIRA", "QUEDA-RIBANCEIRA/OAE"),
        (r"QUEDA-CARGA", "QUEDA-CARGA"),
        (r"^QUEDA$|TABLUDE", "QUEDA-OUTROS"),
        
        (r"LAN.ADO", "OBJETO LANÇADO CONTRA O VEÍCULO"),
        (r"INC.NDIO", "INCÊNDIO"),
        (r"SA.DA DE PISTA", "SAÍDA DE PISTA"),
    ]
    
    for padrao, categoria in regras:
        if re.search(padrao, s):
            return categoria
    
    return "OUTROS/NÃO CLASSIFICADO"

print(list(data["SUBCLASSE"].apply(normalizar_subclasse)))

['CHOQUE-TALUDE/BARRANCO', 'CAPOTAMENTO', 'CHOQUE-DEFENSA/BARREIRA', 'CHOQUE-DEFENSA/BARREIRA', 'ATROP. ANIMAL-OUTROS', 'CHOQUE-DEFENSA/BARREIRA', 'COLISÃO-LATERAL', 'CHOQUE-SINALIZAÇÃO/EQUIPAMENTO', 'TOMBAMENTO', 'TOMBAMENTO', 'OUTROS/NÃO CLASSIFICADO', 'ATROP. ANIMAL-OUTROS', 'CHOQUE-ELEMENTO DE DRENAGEM', 'CHOQUE-SINALIZAÇÃO/EQUIPAMENTO', 'TOMBAMENTO', 'COLISÃO-TRANSVERSAL', 'COLISÃO-TRASEIRA', 'COLISÃO-LATERAL', 'CHOQUE-TALUDE/BARRANCO', 'OUTROS/NÃO CLASSIFICADO', 'OUTROS/NÃO CLASSIFICADO', 'OUTROS/NÃO CLASSIFICADO', 'OUTROS/NÃO CLASSIFICADO', 'CHOQUE-ELEMENTO DE DRENAGEM', 'CHOQUE-SINALIZAÇÃO/EQUIPAMENTO', 'CHOQUE-SINALIZAÇÃO/EQUIPAMENTO', 'COLISÃO-TRASEIRA', 'COLISÃO-TRASEIRA', 'OUTROS/NÃO CLASSIFICADO', 'CHOQUE-ELEMENTO DE DRENAGEM', 'OUTROS/NÃO CLASSIFICADO', 'CHOQUE-POSTE', 'CHOQUE-DEFENSA/BARREIRA', 'OUTROS/NÃO CLASSIFICADO', 'OUTROS/NÃO CLASSIFICADO', 'CHOQUE-SINALIZAÇÃO/EQUIPAMENTO', 'CHOQUE-OUTROS', 'CHOQUE-DEFENSA/BARREIRA', 'COLISÃO-LATERAL', 'CHOQUE-DEFENSA/BARREIRA', '